# In this notebook, we showcase how to use the improve retrieval performance using apply_key_rerotation wrapper.


## Key rerotation compression applies RoPE on the compressed keys (before RoPE). As a consequence, there won't be any "gaps" in the positional embedding.

In [1]:
import numpy as np
import torch
from transformers import pipeline

from kvpress import (
    ExpectedAttentionPress,
    KnormPress,
    ObservedAttentionPress,
    RandomPress,
    SnapKVPress,
    StreamingLLMPress,
    apply_key_rerotation,
)

/mount/home/setup/.cache/pypoetry/virtualenvs/kvpress-PV_RntMw-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load the pipeline and data

In [2]:
# Load pipeline

device = "cuda:0"
ckpt = "meta-llama/Meta-Llama-3.1-8B-Instruct"
attn_implementation = "flash_attention_2"
pipe = pipeline("kv-press-text-generation", model=ckpt, device=device, torch_dtype="auto", model_kwargs={"attn_implementation":attn_implementation})

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  9.69it/s]


In [3]:
import datasets 

df = datasets.load_dataset("simonjegou/ruler", "4096")["test"].to_pandas()
df = df.loc[df["task"] == "niah_single_3"].reset_index(drop=True)

# Use the pipeline with a press

In [4]:
# Pick a press with a compression ratio, you can run the following cells with different presses
compression_ratio = 0.4
press = KnormPress(compression_ratio)

In [5]:
# Run the pipeline on a single question
idx = 3
context = df.iloc[idx]["context"] 
question = df.iloc[idx]["question"] 
true_answer = df.iloc[idx]["answer"][0]

In [6]:
pred_answer = pipe(context, question=question, press=press)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")
print(f"Correctly predicted: {true_answer in pred_answer}")

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


Question:   What is the special magic uuid for flippant-samurai mentioned in the provided text? 
Answer:     47294739-614f-43d7-99db-3ad0ddd1dfb2
Prediction: The special magic uuid for flippant-samurai is: 47294739-614f-43d7-99db-3ad1dfb2
Correctly predicted: False


# Apply apply_key_rerotation with the same overall compression ratio

### Key rerotation can be wrapped around any press

In [7]:
key_rerotation_press = apply_key_rerotation(ExpectedAttentionPress(compression_ratio))
pred_answer = pipe(context, question=question, press=key_rerotation_press)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")
print(f"Correctly predicted: {true_answer in pred_answer}")

Question:   What is the special magic uuid for flippant-samurai mentioned in the provided text? 
Answer:     47294739-614f-43d7-99db-3ad0ddd1dfb2
Prediction: The special magic uuid for flippant-samurai mentioned in the provided text is: 47294739-614f-43d7-99db-3ad0ddd1dfb2
Correctly predicted: True
